In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print("Shape:", df.shape)
print("\nNulls per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes)
print("\nNegative Quantity:", (df['Quantity'] < 0).sum())
print("Negative UnitPrice:", (df['UnitPrice'] < 0).sum())
print("Zero UnitPrice:", (df['UnitPrice'] == 0).sum())

Shape: (541909, 8)

Nulls per column:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows: 5268

Dtypes:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Negative Quantity: 10624
Negative UnitPrice: 2
Zero UnitPrice: 2515


In [4]:
# CustomerID missing hai to us row ka customer-level analysis nahi ho sakta -> drop
df = df.dropna(subset=['CustomerID'])

# Description missing ho to 'Unknown' se fill karo
df['Description'] = df['Description'].fillna('Unknown')

print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [6]:
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [7]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows")

Removed 5225 duplicate rows


In [8]:
# Country names ka whitespace/case consistent karo
df['Country'] = df['Country'].str.strip().str.title()

# CustomerID ko proper integer/string banao
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

# InvoiceDate already datetime honi chahiye
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [9]:
Q1 = df['UnitPrice'].quantile(0.25)
Q3 = df['UnitPrice'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['UnitPrice'] < lower) | (df['UnitPrice'] > upper)]
print("UnitPrice outliers:", outliers.shape[0])

# Decision: cap extreme high prices instead of deleting (rare bulk/wholesale orders)
df['UnitPrice'] = np.where(df['UnitPrice'] > upper, upper, df['UnitPrice'])

UnitPrice outliers: 35802


In [10]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Nulls (CustomerID)', 'Duplicate Rows', 'Negative UnitPrice'],
    'Before': [541909, 135080, 5268, 0],   # apne actual before-cleaning numbers daalo
    'After': [df.shape[0], df['CustomerID'].isnull().sum(), df.duplicated().sum(), (df['UnitPrice'] < 0).sum()]
})
print(summary)

               Metric  Before   After
0                Rows  541909  401604
1  Nulls (CustomerID)  135080       0
2      Duplicate Rows    5268      32
3  Negative UnitPrice       0       0


In [11]:
df.to_csv("Online_Retail_Cleaned.csv", index=False)
print("Cleaned dataset saved.")

Cleaned dataset saved.
